# 1.1 数据源盘点

这一步只看不改：订单明细和 SKU 目录各有哪些字段、每个字段长什么样、有哪些质量问题，
每一类问题数出确切的行数。数清楚了，1.2 才知道要处理什么、处理完该剩多少行。

In [1]:
import json
import sys
import unicodedata
from pathlib import Path

import pandas as pd

import dsflow

ROOT = Path.cwd()
while not (ROOT / "dsflow.yaml").is_file():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from demo_lib import LAST_MONTH, out

OUT = out("1.1")
run = dsflow.start_run("1.1", project=ROOT, hypothesis="订单明细的字段含义清楚，质量问题可以逐项计数")

orders = pd.read_csv(ROOT / "data/raw/orders.csv", dtype={"订单号": str, "SKU": str})
catalog = pd.read_excel(ROOT / "data/raw/sku_catalog.xlsx", dtype={"SKU": str})
run.log_input(ROOT / "data/raw/orders.csv", name="orders_raw", stage="raw")
run.log_input(ROOT / "data/raw/sku_catalog.xlsx", name="sku_catalog", stage="raw")
print(f"订单明细 {len(orders):,} 行 × {orders.shape[1]} 列")
print(f"SKU 目录 {len(catalog):,} 行 × {catalog.shape[1]} 列")
print("订单明细的字段：" + "、".join(orders.columns))


订单明细 121,000 行 × 12 列
SKU 目录 3,000 行 × 7 列
订单明细的字段：订单号、下单时间、采购单位、类目、SKU、商品名称、品牌、数量、单价、金额、渠道、省份


In [2]:
rows = []
for col in orders.columns:
    s = orders[col]
    rows.append({"字段": col, "类型": str(s.dtype), "非空": int(s.notna().sum()), "缺失": int(s.isna().sum()),
                 "不同值": int(s.nunique()), "示例": str(s.dropna().iloc[0])})
field_inventory = pd.DataFrame(rows)
field_inventory.to_csv(OUT / "field_inventory.csv", index=False)
field_inventory


,字段,类型,非空,缺失,不同值,示例
0,订单号,str,121000,0,120000,SO202407035316
1,下单时间,str,121000,0,119905,2024-07-01 00:15:25
2,采购单位,str,121000,0,6,某信息通信公司
3,类目,str,121000,0,3,MRO工业品
4,SKU,str,121000,0,3000,SKU01552
5,商品名称,str,121000,0,4825,六角螺栓 加厚88型
6,品牌,str,121000,0,8,正泰
7,数量,int64,121000,0,126,5
8,单价,float64,121000,0,67254,167.11
9,金额,float64,120359,641,92033,835.55


In [3]:
names = orders["商品名称"].astype(str)
placed = pd.to_datetime(orders["下单时间"])
issues = {
    "订单行": len(orders),
    "完全重复行": int(orders.duplicated().sum()),
    "金额缺失": int(orders["金额"].isna().sum()),
    "数量为负": int((orders["数量"] < 0).sum()),
    "未来日期": int((placed > f"{LAST_MONTH}-30 23:59:59").sum()),
    "商品名称含首尾空白或全角字符": int((names != names.map(lambda x: unicodedata.normalize("NFKC", x).strip())).sum()),
    "SKU目录行": len(catalog),
    "订单中不在目录的SKU": int((~orders["SKU"].isin(catalog["SKU"])).sum()),
}
(OUT / "quality_issues.json").write_text(json.dumps(issues, ensure_ascii=False, indent=1), encoding="utf-8")
run.log_metrics(issues)
run.log_artifact(OUT / "field_inventory.csv", purpose="订单明细逐字段：类型、缺失、不同值、示例", kind="table")
run.log_artifact(OUT / "quality_issues.json", purpose="质量问题逐项计数", kind="table")
print(json.dumps(issues, ensure_ascii=False, indent=1))


{
 "订单行": 121000,
 "完全重复行": 1000,
 "金额缺失": 641,
 "数量为负": 364,
 "未来日期": 2,
 "商品名称含首尾空白或全角字符": 3733,
 "SKU目录行": 3000,
 "订单中不在目录的SKU": 0
}


In [4]:
conclusion = (
    f"订单明细 {len(orders):,} 行 × {orders.shape[1]} 列；完全重复行 {issues['完全重复行']:,}、"
    f"金额缺失 {issues['金额缺失']:,}、数量为负 {issues['数量为负']:,}、未来日期 {issues['未来日期']}，都留给 1.2 处理"
)
run.set_conclusion(conclusion, validity="有效")
run.end()
print(conclusion)


订单明细 121,000 行 × 12 列；完全重复行 1,000、金额缺失 641、数量为负 364、未来日期 2，都留给 1.2 处理
